In [1]:
from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully.
Tokenizer class: BertTokenizer
Vocabulary size: 30522


In [2]:
text = "The football team won the championship."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original text:", text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

Original text: The football team won the championship.
Tokens: ['the', 'football', 'team', 'won', 'the', 'championship', '.']
Token IDs: [1996, 2374, 2136, 2180, 1996, 2528, 1012]


In [3]:
words = [
    "unbelievable",
    "unbelievableness",
    "playing",
    "electroencephalographically"
]

for word in words:
    tokens = tokenizer.tokenize(word)

    print(f"Word: {word}")
    print(f"Tokens: {tokens}")
    print(f"Number of tokens: {len(tokens)}")
    print("-" * 40)

Word: unbelievable
Tokens: ['unbelievable']
Number of tokens: 1
----------------------------------------
Word: unbelievableness
Tokens: ['unbelievable', '##ness']
Number of tokens: 2
----------------------------------------
Word: playing
Tokens: ['playing']
Number of tokens: 1
----------------------------------------
Word: electroencephalographically
Tokens: ['electro', '##ence', '##pha', '##log', '##raphic', '##ally']
Number of tokens: 6
----------------------------------------


In [4]:
text = "The football team won the championship."

encoded = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Input IDs:")
print(encoded["input_ids"])

print("\nAttention Mask:")
print(encoded["attention_mask"])

print("\nInput Shape:")
print(encoded["input_ids"].shape)

Input IDs:
tensor([[ 101, 1996, 2374, 2136, 2180, 1996, 2528, 1012,  102,    0,    0,    0,
            0,    0,    0,    0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]])

Input Shape:
torch.Size([1, 16])


In [5]:
text = "The football team won the championship."

encoded_fixed = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

encoded_dynamic = tokenizer(
    text,
    padding=False,
    truncation=True,
    return_tensors="pt"
)

print("Fixed-length input shape:")
print(encoded_fixed["input_ids"].shape)

print("\nInput shape without fixed-length padding:")
print(encoded_dynamic["input_ids"].shape)

print("\nFixed-length attention mask:")
print(encoded_fixed["attention_mask"])

print("\nAttention mask without fixed-length padding:")
print(encoded_dynamic["attention_mask"])

Fixed-length input shape:
torch.Size([1, 16])

Input shape without fixed-length padding:
torch.Size([1, 9])

Fixed-length attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]])

Attention mask without fixed-length padding:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [6]:
texts = [
    "The football team won the championship.",
    "The stock market fell sharply today.",
    "Scientists discovered a new planet."
]

encoded_batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs:")
print(encoded_batch["input_ids"])

print("\nAttention Mask:")
print(encoded_batch["attention_mask"])

print("\nInput Shape:")
print(encoded_batch["input_ids"].shape)

Input IDs:
tensor([[ 101, 1996, 2374, 2136, 2180, 1996, 2528, 1012,  102],
        [ 101, 1996, 4518, 3006, 3062, 9249, 2651, 1012,  102],
        [ 101, 6529, 3603, 1037, 2047, 4774, 1012,  102,    0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])

Input Shape:
torch.Size([3, 9])


In [7]:
from datasets import load_dataset

dataset = load_dataset("wangrongsheng/ag_news")

sample = dataset["train"][0]

print("Sample:")
print(sample)

print("\nText:")
print(sample["text"])

print("\nLabel:")
print(sample["label"])

Sample:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Text:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Label:
2


In [8]:
from collections import Counter

train_data = dataset["train"]
test_data = dataset["test"]

print("Training samples:", len(train_data))
print("Test samples:", len(test_data))

train_label_counts = Counter(train_data["label"])
test_label_counts = Counter(test_data["label"])

print("\nTraining label counts:")
print(train_label_counts)

print("\nTest label counts:")
print(test_label_counts)

Training samples: 120000
Test samples: 7600

Training label counts:
Counter({2: 30000, 3: 30000, 1: 30000, 0: 30000})

Test label counts:
Counter({2: 1900, 3: 1900, 1: 1900, 0: 1900})


In [9]:
texts = [
    "The football team won the championship.",
    "The stock market fell sharply today.",
    "Scientists discovered a new planet."
]

labels = [1, 2, 3]

encoded_batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs shape:", encoded_batch["input_ids"].shape)
print("Attention Mask shape:", encoded_batch["attention_mask"].shape)

print("\nLabels:")
print(labels)

Input IDs shape: torch.Size([3, 9])
Attention Mask shape: torch.Size([3, 9])

Labels:
[1, 2, 3]


In [10]:
import torch
from torch.utils.data import Dataset

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


texts = [
    "The football team won the championship.",
    "The stock market fell sharply today.",
    "Scientists discovered a new planet."
]

labels = [1, 2, 3]

news_dataset = NewsDataset(texts, labels, tokenizer)

print("Dataset size:", len(news_dataset))
print("\nFirst sample:")
print(news_dataset[0])

Dataset size: 3

First sample:
{'input_ids': tensor([ 101, 1996, 2374, 2136, 2180, 1996, 2528, 1012,  102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1]), 'labels': tensor(1)}


In [11]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    news_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

batch = next(iter(train_loader))

print("Input IDs:")
print(batch["input_ids"])

print("\nAttention Mask:")
print(batch["attention_mask"])

print("\nLabels:")
print(batch["labels"])

print("\nInput IDs shape:", batch["input_ids"].shape)
print("Attention Mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

Input IDs:
tensor([[ 101, 1996, 2374, 2136, 2180, 1996, 2528, 1012,  102],
        [ 101, 1996, 4518, 3006, 3062, 9249, 2651, 1012,  102]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])

Labels:
tensor([1, 2])

Input IDs shape: torch.Size([2, 9])
Attention Mask shape: torch.Size([2, 9])
Labels shape: torch.Size([2])


In [12]:
texts_for_padding = [
    "The football team won the championship.",
    "Scientists discovered a new planet."
]

labels_for_padding = [1, 3]

padding_dataset = NewsDataset(
    texts_for_padding,
    labels_for_padding,
    tokenizer
)

padding_loader = DataLoader(
    padding_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

padding_batch = next(iter(padding_loader))

print("Input IDs:")
print(padding_batch["input_ids"])

print("\nAttention Mask:")
print(padding_batch["attention_mask"])

print("\nInput IDs shape:", padding_batch["input_ids"].shape)
print("Attention Mask shape:", padding_batch["attention_mask"].shape)
print("Labels:", padding_batch["labels"])

Input IDs:
tensor([[ 101, 1996, 2374, 2136, 2180, 1996, 2528, 1012,  102],
        [ 101, 6529, 3603, 1037, 2047, 4774, 1012,  102,    0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])

Input IDs shape: torch.Size([2, 9])
Attention Mask shape: torch.Size([2, 9])
Labels: tensor([1, 3])


In [13]:
from sklearn.model_selection import train_test_split

all_texts = dataset["train"]["text"]
all_labels = dataset["train"]["label"]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    all_texts,
    all_labels,
    test_size=0.1,
    random_state=42,
    stratify=all_labels
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))
print("Test samples:", len(dataset["test"]))

Training samples: 108000
Validation samples: 12000
Test samples: 7600


In [14]:
from collections import Counter

print("Training class distribution:")
print(Counter(train_labels))

print("\nValidation class distribution:")
print(Counter(val_labels))

Training class distribution:
Counter({1: 27000, 2: 27000, 3: 27000, 0: 27000})

Validation class distribution:
Counter({3: 3000, 0: 3000, 2: 3000, 1: 3000})


In [15]:
from sklearn.model_selection import train_test_split

small_train_texts, _, small_train_labels, _ = train_test_split(
    train_texts,
    train_labels,
    train_size=2000,
    random_state=42,
    stratify=train_labels
)

small_val_texts, _, small_val_labels, _ = train_test_split(
    val_texts,
    val_labels,
    train_size=400,
    random_state=42,
    stratify=val_labels
)

print("Small training samples:", len(small_train_texts))
print("Small validation samples:", len(small_val_texts))

Small training samples: 2000
Small validation samples: 400


In [16]:
from collections import Counter

print("Small training distribution:")
print(Counter(small_train_labels))

print("\nSmall validation distribution:")
print(Counter(small_val_labels))

Small training distribution:
Counter({0: 500, 1: 500, 2: 500, 3: 500})

Small validation distribution:
Counter({0: 100, 1: 100, 3: 100, 2: 100})


In [17]:
small_train_dataset = NewsDataset(
    texts=small_train_texts,
    labels=small_train_labels,
    tokenizer=tokenizer
)

small_val_dataset = NewsDataset(
    texts=small_val_texts,
    labels=small_val_labels,
    tokenizer=tokenizer
)

print("Training dataset size:", len(small_train_dataset))
print("Validation dataset size:", len(small_val_dataset))

print("\nFirst training sample:")
print(small_train_dataset[0])

Training dataset size: 2000
Validation dataset size: 400

First training sample:
{'input_ids': tensor([  101, 15768,  3816,  2006,  3514,  7597,  1010,  5618, 17680,  2047,
         2259,  1011,  3020,  3514,  7597,  1998,  6668, 17680,  2015,  2013,
         3316,  2164,  8902,  5867,  1011,  5340, 10893,  3726,  2522,  1012,
         1010,  4895,  9463,  6299, 15492,  1998,  1996,  2047,  2259,  2335,
         2522,  1012,  2741, 15768,  8058,  6928,  1010,  2348,  6627,  6661,
        24356,  2006,  2844,  2836,  1999,  1996, 20681,  4753,  1012,  1012,
         1012,   102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 'labels': tensor(0)}


In [18]:
train_loader = DataLoader(
    small_train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    small_val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 250
Validation batches: 50


In [19]:
train_batch = next(iter(train_loader))

print("Input IDs shape:", train_batch["input_ids"].shape)
print("Attention Mask shape:", train_batch["attention_mask"].shape)
print("Labels shape:", train_batch["labels"].shape)

print("\nLabels:")
print(train_batch["labels"])

print("\nSequence length:")
print(train_batch["input_ids"].shape[1])

Input IDs shape: torch.Size([8, 63])
Attention Mask shape: torch.Size([8, 63])
Labels shape: torch.Size([8])

Labels:
tensor([0, 0, 3, 1, 3, 3, 2, 3])

Sequence length:
63
